# About the Dataset

Dataset: Course Lead Scoring (2026)

Each row is a lead: a person who showed interest in an online course platform. The goal is to predict whether the lead converted, meaning they signed up (converted = 1) or not (converted = 0). This is a binary classification problem.

Categorical features (4)

- lead_source: how the lead found the platform (organic_search, social_media, paid_ads, referral, events)
- industry: the lead’s industry (technology, healthcare, finance, retail, education, manufacturing)
- employment_status: employed, self_employed, unemployed or student
- location: region (north_america, europe, asia, south_america, africa)

Numerical features (4)

- annual_income: yearly income
- number_of_courses_viewed: how many courses the lead browsed
- interaction_count: number of interactions with the platform (emails, visits, etc.)
- lead_score: a pre-computed engagement score between 0 and 1

Target
- converted: 1 if the lead signed up, 0 otherwise

Data quality: Several columns have missing values. Missing categorical values were filled with 'NA' and missing numerical values with 0.0.

# Dataset Download

In [15]:
import pandas as pd
import numpy as np

In [16]:
churn_df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv')

# Data Exploration

In [41]:
print("Shape:", churn_df.shape)
print("\nMissing values:\n", churn_df.isnull().sum())
print("\nConversion rate:", round(churn_df['converted'].mean(), 3))
print("\nNumerical summary:\n", churn_df[numerical].describe().round(2))

Shape: (5000, 9)

Missing values:
 lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

Conversion rate: 0.577

Numerical summary:
        annual_income  number_of_courses_viewed  interaction_count  lead_score
count        5000.00                   5000.00            5000.00     5000.00
mean        50508.66                      2.09               4.47        0.49
std         24927.40                      1.19               2.44        0.20
min             0.00                      0.00               0.00        0.00
25%         35082.75                      1.00               3.00        0.35
50%         52613.00                      2.00               4.00        0.49
75%         69752.75                      3.00               6.00        0.63
max        10

In [40]:
churn_df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NA,0.0,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


# Data Preparation

In [18]:
#Separate categorical and numerical columns
categorical = list(churn_df.dtypes[churn_df.dtypes == 'object'].index)
numerical = list(churn_df.dtypes[churn_df.dtypes != 'object'].index)

# Leave the target out of the feature lists
numerical.remove('converted')

#Fill missing values
churn_df[categorical] = churn_df[categorical].fillna('NA')
churn_df[numerical] = churn_df[numerical].fillna(0.0)

#Confirm nothing is missing anymore
churn_df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [19]:
churn_df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NA,0.0,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Question 1: Most Frequent Observation

In [20]:
churn_df['industry'].mode()

0    technology
Name: industry, dtype: object

In [21]:
churn_df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

## Question 2: Correlation

In [22]:
churn_df[numerical].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [23]:
corr = churn_df[numerical].corr()

pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]

for a, b in pairs:
    print(f"{a} & {b}: {corr.loc[a, b]:.4f}")

interaction_count & lead_score: 0.9157
number_of_courses_viewed & lead_score: 0.7572
number_of_courses_viewed & interaction_count: 0.7216
annual_income & interaction_count: 0.1228


In [25]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(churn_df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

# Pull out the target, then remove it from the feature dataframes
y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [26]:
display(churn_df[numerical].corr())

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


## Question 3: MI

In [27]:
from sklearn.metrics import mutual_info_score

for c in categorical:
    score = mutual_info_score(df_train[c], y_train)
    print(c, round(score, 2))

lead_source 0.03
industry 0.0
employment_status 0.02
location 0.0


In [28]:
mi = {c: round(mutual_info_score(df_train[c], y_train), 2) for c in categorical}
sorted(mi.items(), key=lambda x: x[1], reverse=True)

[('lead_source', 0.03),
 ('employment_status', 0.02),
 ('industry', 0.0),
 ('location', 0.0)]

## Question 4: Logistic Regression

In [29]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

features = categorical + numerical

# One-hot encode: fit on train, then only transform val
dv = DictVectorizer(sparse=False)

train_dicts = df_train[features].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[features].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [30]:
# Train
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [31]:
# Evaluate on validation
y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)
print(round(accuracy, 2))

0.65


## Question 5: Least Useful Feature (Feature Importance)

In [32]:
def train_and_score(features):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_val = dv.transform(df_val[features].to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))

In [33]:
all_features = categorical + numerical
original_acc = train_and_score(all_features)   # not rounded
print("Original:", original_acc)

Original: 0.645


In [34]:
diffs = {}
for f in all_features:
    subset = [c for c in all_features if c != f]
    acc = train_and_score(subset)
    diffs[f] = original_acc - acc

for f, d in sorted(diffs.items(), key=lambda x: x[1]):
    print(f"{f:30s} {d:+.5f}")

annual_income                  -0.07900
industry                       +0.00000
employment_status              +0.00000
location                       +0.00000
lead_score                     +0.00100
number_of_courses_viewed       +0.00200
lead_source                    +0.00300
interaction_count              +0.04400


## Question 6: Regularized Logistic Regression

In [36]:
def train_and_score(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_val = dv.transform(df_val[features].to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))

for C in [0.000001, 0.00001, 0.0001, 0.001]:
    acc = train_and_score(all_features, C=C)
    print(f"C={C:<10} accuracy={round(acc, 3)}")

C=1e-06      accuracy=0.598
C=1e-05      accuracy=0.598
C=0.0001     accuracy=0.613
C=0.001      accuracy=0.645


In [37]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    acc = train_and_score(all_features, C=C)
    print(f"C={C:<10} accuracy={round(acc, 3)}")

C=1e-06      accuracy=0.598
C=1e-05      accuracy=0.598
C=0.0001     accuracy=0.613
C=0.001      accuracy=0.645
